# 🏋️ Model Training & Evaluation — CNNsa-BiLSTM
Notebook ini adalah controller untuk melatih dan mengevaluasi model **CNNsa-BiLSTM** pada dataset BISINDO.

## 1. Setup & Cek Dataset

In [ ]:
import sys
import torch
from pathlib import Path

# Append root path (notebook di Training/CNNsaBiLSTM/ -> naik 2 level ke ROOT)
sys.path.insert(0, str(Path.cwd().parent.parent))

from Training.CNNsaBiLSTM.Config.train_config import verify_data_exists, create_output_dirs
from Training.Scripts.device_utils import get_device

# Pastikan data .npz hasil FeatureExtraction sudah ada
verify_data_exists()
create_output_dirs()

# get_device() print detail GPU (atau warning jelas kalau jatuh ke CPU)
device = get_device()


## 2. Model Summary (Proposed CNNsa-BiLSTM)
Mari kita lihat detail arsitektur model dan jumlah parameter yang digunakan.

In [ ]:
from Training.CNNsaBiLSTM.Models.cnnsa_bilstm import CNNsaBiLSTM
from Training.CNNsaBiLSTM.Config.train_config import MODEL_CONFIG

model = CNNsaBiLSTM(
    input_size=MODEL_CONFIG["input_size"],
    feature_dim=MODEL_CONFIG["feature_dim"],
    hidden_size=MODEL_CONFIG["hidden_size"],
    num_layers=MODEL_CONFIG["num_layers"],
    num_classes=MODEL_CONFIG["num_classes"],
    dropout=MODEL_CONFIG["dropout"],
    attn_reduction=MODEL_CONFIG["attn_reduction"]
).to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"
Trainable parameters: {total_params:,}")


## 3. Training Stage

In [ ]:
from Training.CNNsaBiLSTM.Scripts.trainer import run_training

# Jalankan training loop
run_training()


## 4. Evaluation Stage
Lakukan evaluasi model terbaik pada data test (unseen data) menggunakan CPU untuk benchmark latensi inferensi.

In [ ]:
from Training.CNNsaBiLSTM.Scripts.evaluator import run_evaluation

# Jalankan evaluasi akhir pada test split
run_evaluation()


## 5. Visualisasi Hasil Training

In [ ]:
import json
import matplotlib.pyplot as plt
from Training.CNNsaBiLSTM.Config.train_config import RESULTS_DIR

# Load history
history_path = RESULTS_DIR / "train_history.json"
if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)
        
    epochs = range(1, len(history["train_loss"]) + 1)
    
    plt.figure(figsize=(14, 5))
    
    # Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.title("Loss Curves")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    
    # Accuracy Curve
    plt.subplot(1, 2, 2)
    plt.plot(epochs, [x*100 for x in history["train_acc"]], label="Train Acc")
    plt.plot(epochs, [x*100 for x in history["val_acc"]], label="Val Acc")
    plt.title("Accuracy Curves")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("History training belum tersedia. Silakan jalankan training terlebih dahulu.")


## 6. Tampilkan Confusion Matrix

In [ ]:
from IPython.display import Image, display
from Training.CNNsaBiLSTM.Config.train_config import RESULTS_DIR

cm_path = RESULTS_DIR / "confusion_matrix_cnnsa_bilstm.png"
if cm_path.exists():
    display(Image(filename=str(cm_path)))
else:
    print("Confusion Matrix image belum dibuat.")
